# 01 Data assembly: grid, fault labels, and geophysical features

This notebook builds the analysis grid, the fault labels, and the geophysical feature stack for the GeoDAWN region in western Nevada. Notebooks 02 and 03 read its outputs.

**Inputs.** All inputs are public.

* USGS GeoDAWN airborne magnetic and radiometric grids (ScienceBase).
* USGS Quaternary Fault and Fold Database, file `Qfaults_GIS.zip` (ScienceBase).
* USGS ComCat earthquake catalog, through the FDSN event web service.
* A 3D electrical conductivity model from magnetotelluric (MT) data, from the EarthScope Earth Model Collaboration (EMC) catalog (default: CONUS-MT-2026). No login needed.
* GNSS station velocities: the EarthScope GAGE velocity field (default, needs an EarthScope login), or the Nevada Geodetic Laboratory (NGL) MIDAS velocities.
* Optional: other rasters that you supply, for example isostatic gravity or INGENIOUS layers. Make sure that the license lets you use them.

**Outputs.** The notebook writes these objects under `STORE_ROOT`:

| Object | Content |
|---|---|
| `grid.json` | Grid definition (EPSG:32611, 100 m) |
| `features.zarr` | Feature stack and the `valid` mask |
| `labels.zarr` | Fault raster and distance to the nearest mapped fault |
| `qfaults_aoi.gpkg` | Quaternary fault traces in the area of interest |

**Annotations.**

* `VERIFY:` marks an assumption about an external service or file format. I did not test these against the live services. Check them on the first run.
* `NOTE:` marks a design decision that you can change.

**Behavior on errors.** The notebook stops with a clear error when an input is missing or malformed. It does not fall back silently.

**Environment.** Pangeo-based GeoLab image with `rioxarray`, `geopandas`, `pyogrio`, `scipy`, `requests`, `zarr`, and `fsspec`. Add `s3fs` if `STORE_ROOT` is an S3 URL, `earthscope-sdk` for the EarthScope GNSS velocities, and `netCDF4` (or `h5netcdf`) for the MT model.

#### What the next cell does

It loads the Python libraries that the notebook uses. Nothing is downloaded or computed yet.

- **Standard library.** `io` reads a downloaded text file as if it were a local file. `json` reads and writes the grid definition. `os` and `pathlib` handle file paths and environment variables. `re` matches patterns in file names. `warnings` prints non-fatal warnings. `zipfile` opens downloaded archives.
- **`fsspec`** gives one interface for local folders and cloud storage, so `STORE_ROOT` can be a local folder or an `s3://` address.
- **`geopandas` and `pyogrio`** read and write vector data. Here, that's the fault lines.
- **`rasterio`, `rioxarray`, and `xarray`** read, reproject, and hold gridded data (rasters). Importing `rioxarray` adds the `.rio` methods to xarray objects, which is why it's imported but never called by name.
- **`numpy`, `pandas`, and `scipy`** handle arrays and tables, and provide image filters, interpolation, and nearest-neighbor search.
- **`pyproj.Transformer`** converts longitude and latitude to UTM coordinates in metres.
- **`shapely.geometry.box`** makes the rectangle used to clip the fault lines to the study area.

If this cell fails with `ModuleNotFoundError`, set up the environment as described in the README.

In [ ]:
import io
import json
import os
import pathlib
import re
import warnings
import zipfile

import fsspec
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyogrio
import rasterio
import requests
import rioxarray  # noqa: F401  (registers the .rio accessor)
import xarray as xr
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.transform import from_origin
from rasterio.warp import transform_bounds
from pyproj import Transformer
from scipy import ndimage
from scipy.interpolate import RBFInterpolator
from scipy.spatial import cKDTree
from shapely.geometry import box

## 0. Configuration

Change the values in the next cell. The other cells use these values only.

#### What the next cell does

It only sets values. Every later cell reads its settings from here, so this is the only cell you need to edit.

- **Storage.**
  - `STORE_ROOT` is where the outputs are written. By default, that's a `gems` folder next to the notebooks (`$PWD/gems`): a Jupyter kernel runs in the notebook's own folder. To use a different location, set the `GEMS_STORE` environment variable.
  - `CACHE_DIR` holds the raw downloads, so a re-run doesn't download them again.
- **Grid.** `CRS = "EPSG:32611"` is UTM zone 11 north, a map projection whose coordinates are in metres. `RES = 100.0` makes each grid cell 100 m by 100 m, the cell size of the GEMS challenge.
- **ScienceBase items.** ScienceBase is the USGS data catalog. These IDs identify the GeoDAWN survey and the Quaternary fault database. `QFAULTS_LAYER_FILTER` can restrict which fault layers are used.
- **Area of interest.** `AOI_LONLAT_OVERRIDE` lets you use a smaller area than the whole GeoDAWN survey. This is useful for a quick test.
- **GeoDAWN file selection.** These are regular expressions: text patterns matched against file names. `GEODAWN_INCLUDE` keeps only the GeoTIFF bundles. `GEODAWN_EXCLUDE` drops flight-line data, documents, and metadata. `MIN_VALID_FRACTION` drops rasters that barely overlap the grid.
- **Derivatives.** `POTENTIAL_FIELD_PATTERN` picks the layers that are magnetic or gravity fields. Only those get the edge-detection layers in section 5.
- **Earthquakes.**
  - The service address and the date range.
  - `EQ_MIN_MAG`: the smallest magnitude to include.
  - `EQ_SHALLOW_KM`: the depth limit for "shallow" events.
  - `EQ_SMOOTH_M`: how far each event is spread when its density is mapped (2 km).
- **GNSS.** GNSS means GPS-type stations that measure how the ground moves.
  - `GNSS_SOURCE`: where the station velocities come from.
    - `"earthscope"` (the default) uses the EarthScope GAGE velocity field at `EARTHSCOPE_VEL_URL`, computed by Central Washington University. It needs an EarthScope login.
    - `"midas"` uses the Nevada Geodetic Laboratory file at `MIDAS_URL`, which needs no login.

    The two sources use different reference frames, but that doesn't change the strain rate.
  - `GNSS_MIN_YEARS`, `GNSS_MAX_SIGMA_MM`: quality filters on the stations.
  - `GNSS_STRAIN_STEP_M`: the spacing of the coarse grid on which strain is computed.
  - `GNSS_RBF_SMOOTHING`: how smooth the velocity surface is.
  - `GNSS_MAX_GAP_M`: how far from a station the result is still trusted.
- **`AOI_BUFFER_M`** extends the earthquake and GNSS searches 25 km beyond the grid, so that points just outside it still influence cells near its edge.
- **Magnetotelluric (MT) model.**
  - `EMC_NETCDF_URL` is the EarthScope archive folder for the Earth Model Collaboration (EMC) models.
  - `MT_MODEL_FILE` names the 3D conductivity model to use. It's `None` for no MT layers.
  - `MT_DEPTHS_KM` lists the depths at which to take a horizontal slice. The default, 10 km, is in the middle crust.
- **Extras.** `EXTRA_RASTERS` and `EXTRA_FAULT_VECTORS` let you add your own layers, for example gravity or INGENIOUS faults.

In [ ]:
# Storage. NOTE: a local path works for all three notebooks, because only the notebook
# server writes to the store. Use an S3 URL (s3://bucket/prefix) to share results.
# Default: a "gems" folder next to the notebooks ($PWD/gems; a Jupyter kernel runs in the notebook's folder).
STORE_ROOT = os.environ.get("GEMS_STORE", str(pathlib.Path.cwd() / "gems"))
CACHE_DIR = pathlib.Path(os.environ.get("GEMS_CACHE", "/tmp/gems_cache"))

# Analysis grid. NOTE: same CRS and cell size as the GEMS challenge and the INGENIOUS products.
CRS = "EPSG:32611"
RES = 100.0

# ScienceBase items.
GEODAWN_ITEM = "657e1d85d34e23d3533209f7"   # GeoDAWN magnetic and radiometric surveys
QFAULTS_ITEM = "589097b1e4b072a7ac0cae23"   # Quaternary Fault and Fold Database for the Nation
QFAULTS_FILE = "Qfaults_GIS.zip"
QFAULTS_LAYER_FILTER = None                 # regex on layer name; None = keep every line layer

# Optional AOI override (lon_min, lat_min, lon_max, lat_max). None = GeoDAWN bounding box.
AOI_LONLAT_OVERRIDE = None

# GeoDAWN file selection (regex on file name). None = no filter.
# NOTE: the *_tiffs.zip bundles hold the gridded products as GeoTIFFs. The large *_gdb.zip and *_csv.zip
# files are on ScienceBase's new file manager, which serves an HTML page to scripts rather than the file.
GEODAWN_INCLUDE = r"(?i)_tiffs\.zip$"
GEODAWN_EXCLUDE = r"(?i)(?:flight|line|xyz|csv|pdf|xml|txt|metadata)"
MIN_VALID_FRACTION = 0.01                    # drop layers that cover less than 1 % of the grid

# Layers that get potential-field derivatives (regex on layer name).
POTENTIAL_FIELD_PATTERN = r"(?i)(mag|tmi|rtp|grav|bouguer|iso)"

# Earthquakes: USGS ComCat through the FDSN event service.
FDSN_EVENT_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
FDSN_LIMIT = 20000                           # VERIFY: current per-request limit of the service
EQ_START, EQ_END = "1980-01-01", "2026-09-01"
EQ_MIN_MAG = 1.0
EQ_SHALLOW_KM = 10.0
EQ_SMOOTH_M = 2000.0

# GNSS velocities. "earthscope" = EarthScope GAGE velocity field (CWU solution); needs earthscope-sdk
# and an EarthScope login (run `es login` in a terminal). "midas" = NGL MIDAS velocities; no login.
# The reference frame does not change the strain rate, so either frame works.
GNSS_SOURCE = "earthscope"
# Velocity files are in https://data.earthscope.org/archive/gnss/products/velocity/ (NAM14 = North America fixed).
EARTHSCOPE_VEL_URL = "https://data.earthscope.org/archive/gnss/products/velocity/cwu.snaps_nam14.vel"
# Files in https://geodesy.unr.edu/gps_timeseries/IGS20/midas/ are named by reference frame:
# midas.IGS.txt (IGS20) or plate-fixed versions such as midas.NA.txt (North America fixed).
MIDAS_URL = "https://geodesy.unr.edu/gps_timeseries/IGS20/midas/midas.IGS.txt"
GNSS_MIN_YEARS = 2.5
GNSS_MAX_SIGMA_MM = 1.0
GNSS_STRAIN_STEP_M = 2000.0
GNSS_RBF_SMOOTHING = 1.0      # NOTE: controls strain roughness. Compare with a published strain model.
GNSS_MAX_GAP_M = 50000.0      # strain is masked farther than this from the nearest station
AOI_BUFFER_M = 25000.0        # margin for GNSS and earthquakes around the grid

# Magnetotelluric (MT) conductivity from the EarthScope Earth Model Collaboration (EMC) catalog:
# https://data.earthscope.org/app/products/portal/emc_model_catalog.html. None = no MT layers.
# NOTE: these models resolve features tens of km across. Notebook 03 uses them only to rank candidates.
EMC_NETCDF_URL = "https://data.earthscope.org/archive/seismology/products/emc/netcdf"
MT_MODEL_FILE = "CONUS-MT-2026.r0.2-n4c.nc"   # or "WUS.MT.Bedrosian2021.resistivity.r0.0-n4c.nc"
MT_DEPTHS_KM = (10.0,)   # NOTE: long-period MT resolves little in the top ~1 km; use 5 km or deeper

# Optional extra rasters: {"layer_name": "path or URL that rioxarray can open"}.
EXTRA_RASTERS = {
    # "iso_gravity": "s3://my-bucket/gravity/nv_isostatic_gravity.tif",
}
# Optional extra fault vectors (for example INGENIOUS): {"source_name": "path"}.
EXTRA_FAULT_VECTORS = {
    # "ingenious": "/home/jovyan/data/ingenious_faults.gpkg",
}

#### What the next cell does

It prepares the two storage locations.

1. It creates the cache folder if it doesn't exist.
2. `fsspec.core.url_to_fs` returns a filesystem object for `STORE_ROOT`. The same code then works for a local folder and for S3.
3. It creates the output folder.
4. It defines two helpers:
   - `store_path("name")` builds the full path of an output file.
   - `put_file(local, "name")` copies a local file into the store. Some writers, such as the GeoPackage writer, can only write to a local disk, so the notebook writes those files to the cache first and then copies them.
5. It prints both locations, so you can check where the results will go.

In [ ]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
fs, _ = fsspec.core.url_to_fs(STORE_ROOT)
fs.makedirs(STORE_ROOT, exist_ok=True)

def store_path(rel):
    return f"{STORE_ROOT.rstrip('/')}/{rel}"

def put_file(local, rel):
    fs.put(str(local), store_path(rel))

print("STORE_ROOT:", STORE_ROOT)
print("CACHE_DIR: ", CACHE_DIR)

## 1. Helper functions for ScienceBase and downloads

These functions use the ScienceBase JSON API directly. They do not need `sciencebasepy`.

#### What the next cell does

It defines helper functions for ScienceBase and for downloads. The functions run later; this cell only defines them.

- **`sb_item(item_id)`** asks the ScienceBase API for an item's description, in JSON format. The description includes its title, bounding box, and attached files.
- **`sb_child_ids(item_id)`** lists the item's sub-items. GeoDAWN keeps some of its files in child items. The function stops with an error when it finds 1000 children, because it doesn't request further pages and would otherwise miss files.
- **`sb_files(item_id)`** collects every attached file (name, download address, and size) from the item and, when `recursive=True`, from all of its children.
- **`download(url, dest)`** downloads one file:
  - If the file is already in the cache, it doesn't download it again.
  - It writes the file in 1 MB pieces to `name.part` and renames it only when the download is complete. An interrupted download therefore never looks like a finished file.
  - It stops when the server sends a web page instead of the file. ScienceBase does this for files on its newer file manager.
- **`safe_name(s)`** turns a file name into a lowercase name with underscores, which is used as a layer name.

In [ ]:
SB_API = "https://www.sciencebase.gov/catalog"

def sb_item(item_id):
    r = requests.get(f"{SB_API}/item/{item_id}", params={"format": "json"}, timeout=60)
    r.raise_for_status()
    return r.json()

def sb_child_ids(item_id, page=1000):
    # VERIFY: ScienceBase paging. The function stops if one page is full.
    r = requests.get(f"{SB_API}/items",
                     params={"parentId": item_id, "format": "json", "max": page, "fields": "id"},
                     timeout=60)
    r.raise_for_status()
    items = r.json().get("items", [])
    if len(items) >= page:
        raise RuntimeError(f"Item {item_id} has {page} or more children. Add paging to sb_child_ids.")
    return [it["id"] for it in items]

def sb_files(item_id, recursive=True):
    it = sb_item(item_id)
    files = list(it.get("files", []))
    for facet in it.get("facets", []):          # VERIFY: some items keep files under "facets"
        files += facet.get("files", [])
    out = []
    for f in files:
        url = f.get("url") or f.get("downloadUri")
        if url:
            out.append({"item": item_id, "item_title": it.get("title", ""),
                        "name": f.get("name", ""), "url": url, "size": f.get("size")})
    if recursive:
        for cid in sb_child_ids(item_id):
            out += sb_files(cid, recursive=True)
    return out

def download(url, dest):
    dest = pathlib.Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_name(dest.name + ".part")
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        # ScienceBase can answer 200 with an HTML page (login, landing or error page) instead of the file.
        ctype = r.headers.get("Content-Type", "")
        if "text/html" in ctype:
            raise RuntimeError(f"{url} returned an HTML page ({ctype}), not a file. Open it in a browser to see why.")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    tmp.rename(dest)
    return dest

def safe_name(s):
    return re.sub(r"[^0-9a-zA-Z]+", "_", s).strip("_").lower()

## 2. Analysis grid

The grid covers the GeoDAWN bounding box, projected to UTM zone 11N and snapped to 100 m. The `valid` mask (section 7) marks the cells inside the GeoDAWN survey coverage. Notebook 03 trains and predicts only inside that mask.

#### What the next cell does

It defines the 100 m analysis grid. Every layer in all three notebooks is placed on this grid, so a given cell covers the same patch of ground in every layer.

1. **Area.** It uses `AOI_LONLAT_OVERRIDE` if you set one. Otherwise it reads the bounding box of the GeoDAWN survey from ScienceBase, in longitude and latitude.
2. **Convert to metres.** `transform_bounds` converts the box to UTM zone 11 coordinates. A straight edge in longitude/latitude becomes slightly curved in UTM, so the function checks 51 points along each edge rather than only the corners. That way the box still covers the whole area.
3. **Snap to 100 m.** It rounds the edges outward to whole multiples of 100 m, so the cells line up with the GEMS and INGENIOUS grids.
4. **Size and position.** `W` and `H` are the number of columns and rows. `TRANSFORM` is the rule that turns a row and column into map coordinates: it starts at the top-left corner and moves 100 m per cell.
5. **Save the grid.** `GRID` holds the grid definition. It's saved as `grid.json`, and notebooks 02 and 03 read it from there.
6. **Template.** `template` is an empty grid, filled with NaN ("no value"), with the coordinates of every cell centre. Every raster is later resampled onto this template, so all layers line up exactly.
7. **Search area for points.** It computes a second box, 25 km larger on each side, for the earthquake and GNSS searches. `to_utm` converts longitude and latitude of those points to metres.
8. It prints the grid size.

In [ ]:
if AOI_LONLAT_OVERRIDE is not None:
    AOI_LONLAT = tuple(AOI_LONLAT_OVERRIDE)
else:
    bb = sb_item(GEODAWN_ITEM).get("spatial", {}).get("boundingBox")
    if not bb:
        raise RuntimeError("The GeoDAWN item has no bounding box. Set AOI_LONLAT_OVERRIDE.")
    AOI_LONLAT = (bb["minX"], bb["minY"], bb["maxX"], bb["maxY"])
print("AOI (lon/lat):", AOI_LONLAT)

xmin, ymin, xmax, ymax = transform_bounds("EPSG:4326", CRS, *AOI_LONLAT, densify_pts=51)
xmin, ymin = float(np.floor(xmin / RES) * RES), float(np.floor(ymin / RES) * RES)
xmax, ymax = float(np.ceil(xmax / RES) * RES), float(np.ceil(ymax / RES) * RES)
W, H = int(round((xmax - xmin) / RES)), int(round((ymax - ymin) / RES))
TRANSFORM = from_origin(xmin, ymax, RES, RES)
GRID = {"crs": CRS, "res": RES, "xmin": xmin, "ymax": ymax, "width": W, "height": H,
        "aoi_lonlat": [float(v) for v in AOI_LONLAT]}

def make_template(grid):
    x = grid["xmin"] + grid["res"] * (np.arange(grid["width"]) + 0.5)
    y = grid["ymax"] - grid["res"] * (np.arange(grid["height"]) + 0.5)
    da = xr.DataArray(np.full((grid["height"], grid["width"]), np.nan, np.float32),
                      coords={"y": y, "x": x}, dims=("y", "x"))
    tr = from_origin(grid["xmin"], grid["ymax"], grid["res"], grid["res"])
    return da.rio.write_crs(grid["crs"]).rio.write_transform(tr)

template = make_template(GRID)
with fs.open(store_path("grid.json"), "w") as f:
    json.dump(GRID, f, indent=2)

# Buffered AOI in lon/lat for the point data queries.
AOI_BUF_LONLAT = transform_bounds(CRS, "EPSG:4326", xmin - AOI_BUFFER_M, ymin - AOI_BUFFER_M,
                                  xmax + AOI_BUFFER_M, ymax + AOI_BUFFER_M, densify_pts=51)
to_utm = Transformer.from_crs("EPSG:4326", CRS, always_xy=True)
print(f"Grid: {W} x {H} cells ({W * H / 1e6:.1f} M), {W * H * RES * RES / 1e6:,.0f} km2")

## 3. Fault labels

The Quaternary Fault and Fold Database is the label source. It is incomplete and some traces are offset from the true surface trace. Notebook 03 treats unmapped cells as unlabeled, not as confirmed negatives. For that reason, this section also saves the distance from every cell to the nearest mapped fault.

#### What the next cell does

It downloads the USGS Quaternary Fault and Fold Database and finds the fault lines in it. These mapped faults are the training labels.

1. It finds `Qfaults_GIS.zip` among the files of the ScienceBase item, and stops if it isn't there.
2. It downloads the archive, unless it's already in the cache, and unzips it once.
3. It searches the unzipped folder for vector files: shapefiles (`.shp`), GeoPackages (`.gpkg`), and file geodatabases (`.gdb`).
4. `pyogrio.list_layers` lists the layers in each file with their geometry type. The cell keeps only layers of **lines**, because faults are mapped as lines and the archive can also hold other kinds of layers. If you set `QFAULTS_LAYER_FILTER`, only the layers whose names match it are kept.
5. It prints the layers it will use. With more than one layer, it warns you. Two layers can hold the same faults, for example detailed and simplified versions, and the faults would then count twice.

In [ ]:
qf = [f for f in sb_files(QFAULTS_ITEM, recursive=False) if f["name"] == QFAULTS_FILE]
if not qf:
    raise RuntimeError(f"{QFAULTS_FILE} is not attached to ScienceBase item {QFAULTS_ITEM}.")
qf_zip = download(qf[0]["url"], CACHE_DIR / QFAULTS_FILE)
qf_dir = CACHE_DIR / "qfaults"
if not qf_dir.exists():
    with zipfile.ZipFile(qf_zip) as z:
        z.extractall(qf_dir)

sources = [p for p in qf_dir.rglob("*") if p.suffix.lower() in (".shp", ".gpkg", ".gdb")]
line_layers = []
for src in sources:
    for name, gtype in pyogrio.list_layers(src):
        if gtype and "line" in str(gtype).lower():
            if QFAULTS_LAYER_FILTER is None or re.search(QFAULTS_LAYER_FILTER, name):
                line_layers.append((src, name))
if not line_layers:
    raise RuntimeError(f"No line layers found in {qf_dir}. Inspect the archive.")
print("Line layers used:")
for src, name in line_layers:
    print(f"  {src.name} :: {name}")
if len(line_layers) > 1:
    # VERIFY: if the archive holds detailed and simplified traces, keep only the detailed layer.
    warnings.warn("More than one line layer is used. Set QFAULTS_LAYER_FILTER to avoid duplicates.")

#### What the next cell does

It clips the fault lines to the grid, saves them, and turns them into a raster (a grid of values) that the model can learn from.

1. **`load_lines`** reads one layer and:
   - stops if the layer has no coordinate system,
   - converts it to the grid's UTM coordinates,
   - keeps only the lines that touch the grid, and cuts them at the grid edge,
   - lowercases the column names. GeoPackage treats `FAULT_NAME` and `fault_name` as the same column, so mixed spellings from different layers would clash when the file is written.
   - records where each line came from, in `label_source`.
2. **Combine and save.** It combines all Quaternary fault layers, adds any `EXTRA_FAULT_VECTORS`, and saves the result as `qfaults_aoi.gpkg`. It writes the file to the cache and then copies it to the store.
3. **Rasterize.** `burn` sets each 100 m cell that a fault line crosses to 1 and all other cells to 0. `all_touched=True` marks every cell the line touches, not only the cells whose centre lies on the line, so thin lines don't break into gaps.
   - `fault_q` holds the Quaternary faults only.
   - `fault_all` also includes any extra fault sources.
4. **Distance to faults.** `distance_transform_edt` computes, for every cell, the straight-line distance to the nearest fault cell. Multiplying by 100 converts it to metres. Notebook 03 uses this distance to decide which cells are far enough from mapped faults to use as "probably not a fault" examples.
5. It prints how many cells contain a fault.

In [ ]:
aoi_box = box(xmin, ymin, xmax, ymax)

def load_lines(path, layer=None, source=""):
    g = gpd.read_file(path, layer=layer)
    if g.crs is None:
        raise RuntimeError(f"{path} has no CRS.")
    g = g.to_crs(CRS)
    g = g[g.intersects(aoi_box)].copy()
    g["geometry"] = g.geometry.intersection(aoi_box)
    g = g[~g.geometry.is_empty]
    # GeoPackage field names are case-insensitive: "FAULT_NAME" and "fault_name" from
    # different layers would collide on write. Lowercase them and keep the first of any duplicates.
    g = g.rename_geometry("geometry") if g.geometry.name != "geometry" else g
    g.columns = [c.lower() for c in g.columns]
    g = g.loc[:, ~g.columns.duplicated()]
    g["label_source"] = source
    return g

qfaults = pd.concat([load_lines(s, n, "qfaults") for s, n in line_layers], ignore_index=True)
qfaults = gpd.GeoDataFrame(qfaults, geometry="geometry", crs=CRS)
print(len(qfaults), "Quaternary fault features in the AOI")

extra = [load_lines(p, None, k) for k, p in EXTRA_FAULT_VECTORS.items()]
all_faults = gpd.GeoDataFrame(pd.concat([qfaults] + extra, ignore_index=True), geometry="geometry", crs=CRS)

local = CACHE_DIR / "qfaults_aoi.gpkg"
all_faults.to_file(local, driver="GPKG")
put_file(local, "qfaults_aoi.gpkg")

def burn(gdf):
    if len(gdf) == 0:
        return np.zeros((H, W), np.uint8)
    # NOTE: all_touched=True marks every cell that a trace crosses.
    return rasterize(((g, 1) for g in gdf.geometry), out_shape=(H, W), transform=TRANSFORM,
                     fill=0, all_touched=True, dtype="uint8")

fault_q = burn(qfaults)
fault_all = burn(all_faults)
fault_dist_m = (ndimage.distance_transform_edt(fault_all == 0) * RES).astype(np.float32)
print(f"Fault cells: {fault_all.sum():,} ({fault_all.mean() * 100:.2f} % of the grid)")

## 4. GeoDAWN grids

The next cell lists all files on the GeoDAWN item and its child items. Look at the table, then set `GEODAWN_INCLUDE` and `GEODAWN_EXCLUDE` in section 0 if necessary.

`VERIFY:` the GeoDAWN grid format. The code opens every file that GDAL can read as a single-band raster. If the grids use a format that GDAL cannot read (for example some Geosoft formats), the cell stops. Convert those grids to GeoTIFF first.

#### What the next cell does

It lists the GeoDAWN files and chooses which ones to download.

1. `sb_files(GEODAWN_ITEM, recursive=True)` collects every file on the GeoDAWN item and its child items. The table shows each file's name and size, so you can check the selection.
2. It keeps files with extensions that can hold rasters, such as `.tif` and `.zip`.
3. It removes the files that match `GEODAWN_EXCLUDE`, for example flight-line data, documents, and metadata.
4. It keeps only the files that match `GEODAWN_INCLUDE`, by default the `*_tiffs.zip` bundles.
5. It prints how many files are selected and their total size, which is roughly how much the next cell will download.

If a file you need isn't selected, change the two patterns in section 0 and run this cell again.

In [ ]:
gd_files = pd.DataFrame(sb_files(GEODAWN_ITEM, recursive=True))
if gd_files.empty:
    raise RuntimeError("No files found on the GeoDAWN item.")
print(len(gd_files), "files on the GeoDAWN item tree")
display(gd_files[["item_title", "name", "size"]])

RASTER_EXT = (".tif", ".tiff", ".zip", ".grd", ".ers", ".img", ".gxf")
sel = gd_files[gd_files["name"].str.lower().str.endswith(RASTER_EXT)]
if GEODAWN_EXCLUDE:
    sel = sel[~sel["name"].str.contains(GEODAWN_EXCLUDE, regex=True)]
if GEODAWN_INCLUDE:
    sel = sel[sel["name"].str.contains(GEODAWN_INCLUDE, regex=True)]
print(f"Selected {len(sel)} files, {pd.to_numeric(sel['size']).fillna(0).sum() / 1e9:.2f} GB")

#### What the next cell does

It downloads the selected GeoDAWN files, finds the rasters inside them, and resamples each raster onto the 100 m grid.

- **`readable_rasters(path)`** finds the usable rasters in one downloaded file:
  - For a zip file, it first checks that the file really is a zip. If it isn't, for example because the server sent a web page, it deletes the cached copy, so the next run downloads it again, and stops with an error that shows the start of the file. Otherwise it unzips the archive once.
  - It skips files that are never rasters, such as `.xml`, `.pdf`, and shapefile parts.
  - It tries to open every other file with `rasterio` and keeps the single-band rasters larger than 10 by 10 cells. This drops multi-band images such as map figures and thumbnails.
- **`to_grid(src, name)`** resamples one raster onto the 100 m grid:
  - It opens the raster with its "no data" value turned into NaN, and removes the band dimension.
  - It stops if the raster has no coordinate system.
  - It estimates the raster's cell size in metres. For a raster in degrees, it multiplies by about 111 km per degree.
  - It chooses how to resample. If the source is finer than 100 m, many source cells fall into each grid cell, so it takes their **average**. If the source is coarser, it **interpolates** between neighboring source cells (bilinear).
  - `reproject_match` puts the result exactly onto the template grid. The cell then copies the template's coordinates onto the result, so that tiny floating-point differences can't stop the layers from combining later.
- **The loop** downloads each selected file and handles every raster in it:
  - It names the layer `gd_` plus a cleaned-up file name, and adds `_b` if that name is already taken.
  - It resamples the raster and measures the fraction of the grid it covers. It skips rasters that cover less than `MIN_VALID_FRACTION`, typically grids from other survey areas.
  - It stores the rest in `layers` and prints each one with its coverage.
  - It stops if no raster could be read.

In [ ]:
SKIP_EXT = {".xml", ".txt", ".pdf", ".csv", ".prj", ".aux", ".ovr", ".xyz", ".dbf", ".shx", ".shp", ".html", ".htm"}

def readable_rasters(path):
    path = pathlib.Path(path)
    if path.suffix.lower() == ".zip":
        out_dir = path.with_suffix("")
        if not zipfile.is_zipfile(path):
            size, head = path.stat().st_size, path.read_bytes()[:200]
            path.unlink()   # remove the bad cached copy so the next run downloads it again
            raise RuntimeError(f"{path} is not a zip file ({size:,} bytes, cached copy deleted). "
                               f"First bytes: {head!r}")
        if not out_dir.exists():
            with zipfile.ZipFile(path) as z:
                z.extractall(out_dir)
        paths = [p for p in out_dir.rglob("*") if p.is_file()]
    else:
        paths = [path]
    found = []
    for p in paths:
        if p.suffix.lower() in SKIP_EXT:
            continue
        try:
            with rasterio.open(p) as src:
                if src.count == 1 and src.width > 10 and src.height > 10:
                    found.append(p)
        except rasterio.errors.RasterioIOError:
            pass
    return found

def to_grid(src, name):
    da = rioxarray.open_rasterio(src, masked=True)
    if "band" in da.dims:
        da = da.squeeze("band", drop=True)
    if da.rio.crs is None:
        raise RuntimeError(f"{name}: {src} has no CRS. Assign one with .rio.write_crs() first.")
    res = abs(da.rio.resolution()[0])
    res_m = res * 111_000.0 if da.rio.crs.is_geographic else res   # VERIFY: projected units are metres
    # NOTE: average when the source is finer than the grid, bilinear when it is coarser.
    method = Resampling.average if res_m < RES else Resampling.bilinear
    out = da.astype("float32").rio.reproject_match(template, resampling=method)
    out = out.assign_coords(x=template.x, y=template.y)
    return out.rename(name)

layers = {}
geodawn_names = []
for _, row in sel.iterrows():
    local = download(row["url"], CACHE_DIR / "geodawn" / row["item"] / row["name"])
    for p in readable_rasters(local):
        name = "gd_" + safe_name(p.stem)
        while name in layers:
            name += "_b"
        da = to_grid(p, name)
        frac = float(np.isfinite(da.values).mean())
        if frac < MIN_VALID_FRACTION:
            print(f"  skip {name}: {frac:.3%} valid")
            continue
        layers[name] = da
        geodawn_names.append(name)
        print(f"  {name:45s} {frac:6.1%} valid   from {p.name}")
if not geodawn_names:
    raise RuntimeError("No readable GeoDAWN rasters. Check the file table and the selection patterns.")

## 5. Extra rasters and potential-field derivatives

The derivatives are the horizontal gradient magnitude (`thg`), the first vertical derivative (`vdr`, FFT method), and the tilt angle (`tilt`). Maxima of `thg` and zero crossings of `tilt` often mark buried contacts and faults that have no surface scarp.

#### What the next cell does

It adds your own rasters, if any, and then computes edge-detection layers for the magnetic and gravity grids.

1. **Extra rasters.** Each entry in `EXTRA_RASTERS` is resampled onto the grid with `to_grid`, in the same way as the GeoDAWN rasters.
2. **Why derivatives.** Magnetic and gravity maps are smooth. The edges of buried rock bodies, which often lie along faults, show up as places where the field changes quickly. The three derivatives make those edges easier to see.
3. **`potential_field_derivatives`** works on one layer:
   - **Fill gaps.** The Fourier transform below can't handle NaN, so each gap cell temporarily takes the value of its nearest valid neighbor. The average is subtracted.
   - **Pad.** It mirrors 64 cells of data around the edges. The Fourier method treats the map as if it wrapped around, like a tile pattern, and the padding stops the left edge from affecting the right edge.
   - **Vertical derivative (`vdr`).** This is how fast the field changes with height. In the Fourier domain (a description of the map as a sum of waves), it is computed by multiplying each wave by its wavenumber, which is how tightly it oscillates. Short, sharp features, from shallow sources, therefore become stronger. The padding is then cut off again.
   - **Horizontal gradient (`thg`).** `np.gradient` gives the rate of change east–west and north–south, and `thg` is its magnitude: how steeply the field changes sideways. Its maxima lie over the edges of rock bodies.
   - **Tilt angle (`tilt`).** `arctan(vdr / thg)`. It's a normalized value between −90° and +90°, and it is close to zero over edges, whether the anomaly is strong or weak. This lets weak and strong anomalies be compared.
   - **Restore gaps.** It sets the filled gap cells back to NaN.
4. **The loop** applies this to every layer whose name matches `POTENTIAL_FIELD_PATTERN`, and adds the layers `<name>_thg`, `<name>_vdr`, and `<name>_tilt`. Radiometric layers don't match the pattern, so they get no derivatives.

In [ ]:
for name, src in EXTRA_RASTERS.items():
    layers[name] = to_grid(src, name)
    print(f"  {name}: {float(np.isfinite(layers[name].values).mean()):.1%} valid")

def potential_field_derivatives(z, dx, pad=64):
    """Horizontal gradient magnitude, first vertical derivative, and tilt angle.

    NOTE: NaN cells are filled with the nearest valid value before the FFT.
    Results are set back to NaN at those cells. Expect edge effects near large gaps.
    """
    valid = np.isfinite(z)
    if valid.sum() < 100:
        raise ValueError("Too few valid cells for derivatives.")
    idx = ndimage.distance_transform_edt(~valid, return_distances=False, return_indices=True)
    zf = z[tuple(idx)].astype(np.float64)
    zf -= zf.mean()
    zp = np.pad(zf, pad, mode="reflect")
    ny, nx = zp.shape
    kx = 2 * np.pi * np.fft.fftfreq(nx, d=dx)
    ky = 2 * np.pi * np.fft.fftfreq(ny, d=dx)
    kk = np.hypot(*np.meshgrid(kx, ky))
    vdr = np.real(np.fft.ifft2(np.fft.fft2(zp) * kk))[pad:-pad, pad:-pad]
    gr, gc = np.gradient(zf, dx)
    thg = np.hypot(gr, gc)
    tilt = np.arctan2(vdr, thg)
    return {k: np.where(valid, v, np.nan).astype(np.float32)
            for k, v in {"thg": thg, "vdr": vdr, "tilt": tilt}.items()}

for name in [n for n in list(layers) if re.search(POTENTIAL_FIELD_PATTERN, n)]:
    for suffix, arr in potential_field_derivatives(layers[name].values, RES).items():
        layers[f"{name}_{suffix}"] = template.copy(data=arr).rename(f"{name}_{suffix}")
    print("  derivatives for", name)

## 6. Earthquake density and GNSS strain rate

**Earthquakes.** The query splits a time window in two when the service returns its maximum number of events. So large aftershock sequences do not truncate the catalog without notice. The result is two log-density layers: all events, and events shallower than `EQ_SHALLOW_KM`.

**GNSS.** The station velocities come from the EarthScope GAGE velocity field (`GNSS_SOURCE = "earthscope"`) or from NGL MIDAS (`"midas"`). The east and north velocities are interpolated with a smoothed thin-plate spline. The strain rate tensor comes from the spatial gradients of that field. The reference frame does not change the strain rate, because a rigid rotation adds rotation only. `NOTE:` this is a simple estimator. Compare its output with a published strain rate model for the Great Basin before you rely on it.

#### What the next cell does

It downloads the earthquake catalog from USGS ComCat.

- **`fetch_comcat`** asks the USGS event service for a CSV table of earthquakes in the search box, the time window, and above the minimum magnitude.
  - An answer of HTTP 204 means "no events", and returns an empty table.
  - If the answer has as many rows as the service limit (`FDSN_LIMIT`), the list was probably cut off. The function then splits the time window in half and asks for each half separately. It repeats this until each part fits, so large aftershock sequences aren't silently truncated.
- **The loop** sends one request per calendar year, from `EQ_START` to `EQ_END`, so that most requests stay under the limit.
- It combines the years and removes events that appear twice. An event exactly on the boundary between two windows can be returned by both.
- It converts each event's longitude and latitude to UTM `x` and `y` in metres.
- It stops if no events were returned, and otherwise prints the number of events.

In [ ]:
def fetch_comcat(bbox_lonlat, t0, t1, min_mag):
    lon0, lat0, lon1, lat1 = bbox_lonlat
    t0, t1 = pd.Timestamp(t0), pd.Timestamp(t1)
    params = {"format": "csv", "starttime": t0.isoformat(), "endtime": t1.isoformat(),
              "minlatitude": lat0, "maxlatitude": lat1, "minlongitude": lon0, "maxlongitude": lon1,
              "minmagnitude": min_mag, "orderby": "time-asc", "limit": FDSN_LIMIT}
    r = requests.get(FDSN_EVENT_URL, params=params, timeout=300)
    if r.status_code == 204 or not r.text.strip():
        return pd.DataFrame()
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text))
    if len(df) >= FDSN_LIMIT:
        mid = t0 + (t1 - t0) / 2
        return pd.concat([fetch_comcat(bbox_lonlat, t0, mid, min_mag),
                          fetch_comcat(bbox_lonlat, mid, t1, min_mag)], ignore_index=True)
    return df

eq_parts = []
for y0 in range(pd.Timestamp(EQ_START).year, pd.Timestamp(EQ_END).year + 1):
    t0 = max(pd.Timestamp(f"{y0}-01-01"), pd.Timestamp(EQ_START))
    t1 = min(pd.Timestamp(f"{y0 + 1}-01-01"), pd.Timestamp(EQ_END))
    if t1 > t0:
        eq_parts.append(fetch_comcat(AOI_BUF_LONLAT, t0, t1, EQ_MIN_MAG))
eq = pd.concat(eq_parts, ignore_index=True).drop_duplicates("id")
if eq.empty:
    raise RuntimeError("No earthquakes returned. Check the AOI and the service.")
eq["x"], eq["y"] = to_utm.transform(eq["longitude"].values, eq["latitude"].values)
print(f"{len(eq):,} events, M >= {EQ_MIN_MAG}, {EQ_START} to {EQ_END}")

#### What the next cell does

It defines `point_log_density`, which turns a list of earthquake locations into a smooth map of earthquake density. It only defines the function; the next cell uses it.

1. **Padded grid.** It uses a grid that extends three smoothing distances (`sigma_m`) beyond each side of the analysis grid. Events just outside the grid still add to the density near its edge.
2. **Count.** It works out which cell each event falls into. `np.add.at` adds 1 per event, and counts correctly when several events fall into the same cell.
3. **Smooth.** A Gaussian filter spreads each count over the surrounding cells: most of it within 2 km, less farther away. The result is a smooth density surface rather than isolated dots.
4. **Crop and convert.** It removes the padding and divides by the cell area in km², so the values are events per km².
5. **Logarithm.** It returns `log10(density + 0.001)`. Densities range over several orders of magnitude, so the logarithm makes them easier to compare. The small `+ 0.001` avoids the logarithm of zero in cells with no events.

In [ ]:
def point_log_density(x, y, sigma_m, grid):
    """log10 of kernel-smoothed point density (events per km2) on the grid."""
    res, h, w = grid["res"], grid["height"], grid["width"]
    pad = int(np.ceil(3 * sigma_m / res))
    col = np.floor((np.asarray(x) - grid["xmin"]) / res).astype(int) + pad
    row = np.floor((grid["ymax"] - np.asarray(y)) / res).astype(int) + pad
    ok = (col >= 0) & (col < w + 2 * pad) & (row >= 0) & (row < h + 2 * pad)
    counts = np.zeros((h + 2 * pad, w + 2 * pad), np.float64)
    np.add.at(counts, (row[ok], col[ok]), 1.0)
    dens = ndimage.gaussian_filter(counts, sigma=sigma_m / res, mode="constant")
    dens = dens[pad:pad + h, pad:pad + w] / (res * res / 1e6)
    return np.log10(dens + 1e-3).astype(np.float32)

#### What the next cell does

It creates two earthquake density layers with the function from the previous cell:

- **`eq_logdens_all`**: all events.
- **`eq_logdens_shallow`**: only events at most `EQ_SHALLOW_KM` (10 km) deep. Shallow earthquakes are more likely to be on faults that reach close to the surface, and those faults matter most for geothermal fluid flow.

`template.copy(data=...)` puts each array onto the template grid, so the layer gets the same coordinates as all the others. The cell then prints the number of shallow events.

In [ ]:
layers["eq_logdens_all"] = template.copy(data=point_log_density(eq.x, eq.y, EQ_SMOOTH_M, GRID)).rename("eq_logdens_all")
shallow = eq[eq["depth"] <= EQ_SHALLOW_KM]
layers["eq_logdens_shallow"] = template.copy(data=point_log_density(shallow.x, shallow.y, EQ_SMOOTH_M, GRID)).rename("eq_logdens_shallow")
print(f"{len(shallow):,} events shallower than {EQ_SHALLOW_KM} km")

#### What the next cell does

It downloads the GNSS station velocities and keeps the reliable stations in and around the study area. `GNSS_SOURCE` chooses the source.

**EarthScope (`load_earthscope_velocities`, the default).** It reads the GAGE velocity file from the EarthScope data archive.
1. **Login.** The archive needs an EarthScope login. `earthscope-sdk` reads the login that `es login` saved on this server, renews the access token if it's about to expire, and the cell sends the token with the download. If there's no valid login, the cell stops and tells you to run `es login` in a terminal.
2. **Check the download.** It stops if the server sent a web page instead of the file, which usually means the login or the address is wrong.
3. **Find the columns.** The file starts with a text description of each field, followed by a line beginning with `*Dot#` that names the 30 columns. The cell reads the column names from that line.
4. **Read the rows.** The first value on each line is the 4-character station code (`Dot#`), and the last 28 are numbers. Everything in between is the station name, which can contain spaces, for example "Dover, Delaware". Splitting the line this way keeps such names in one piece.
5. **One line per station.** A station has one line per position segment. The record is split wherever an antenna change or an earthquake shifted the station's position. All segments of a station share the same velocity. The cell keeps one line per station, and measures the record from the start of its first segment to the end of its last. Measuring only the last segment would make long-running stations look short-lived.
6. **Units check.** Velocities and their uncertainties are in metres per year. If the values look like millimetres per year instead, the cell stops.

**MIDAS (`load_midas_velocities`).** This source needs no login. It reads the Nevada Geodetic Laboratory file, a whitespace-separated table with 27 columns (named in `MIDAS_COLS`), and stops if the file has fewer columns.

**For both sources:**
1. **Common layout.** Both loaders return the same columns: station, latitude, longitude, east and north velocity, their uncertainties (m/yr), and record length (years).
2. **Longitudes.** It converts longitudes into the range −180 to 180. Both files list western longitudes as values like 240° or −229°.
3. **Latitude check.** It checks that the latitudes are real latitudes.
4. **Filter.** It keeps the stations that are:
   - inside the search box, which is 25 km larger than the grid,
   - measured for at least `GNSS_MIN_YEARS` (2.5 years), because short records give unreliable velocities,
   - uncertain by at most `GNSS_MAX_SIGMA_MM` (1 mm/yr) in both the east and the north velocity.
5. **Coordinates.** It converts the stations' longitudes and latitudes to UTM metres.
6. **Minimum count.** It stops if fewer than 10 stations remain, because too few stations can't define a strain field.

In [ ]:
def load_earthscope_velocities(url):
    """EarthScope GAGE velocity file (.vel). Velocities and sigmas in m/yr; epochs as YYYYMMDDhhmmss."""
    try:
        from earthscope_sdk import EarthScopeClient
    except ImportError as e:
        raise RuntimeError("GNSS_SOURCE = 'earthscope' needs earthscope-sdk: run %pip install earthscope-sdk, "
                           "or set GNSS_SOURCE = 'midas'.") from e
    auth = EarthScopeClient().ctx.auth_flow
    try:
        auth.refresh_if_necessary()
        token = auth.access_token
    except Exception as e:
        raise RuntimeError("No valid EarthScope login. Open a terminal, run `es login`, and run this cell again. "
                           "Or set GNSS_SOURCE = 'midas'.") from e
    r = requests.get(url, headers={"Authorization": f"Bearer {token}"}, timeout=300)
    r.raise_for_status()
    if "text/html" in r.headers.get("Content-Type", ""):
        raise RuntimeError(f"{url} returned a web page, not a velocity file. Check the URL and your EarthScope login.")

    # The file has free-text header lines, then a "*Dot#" line that names the 30 columns.
    lines = r.text.splitlines()
    hdr_i = next((i for i, l in enumerate(lines) if l.lstrip().startswith("*Dot#")), None)
    if hdr_i is None:
        raise RuntimeError(f"No '*Dot#' column header in {url}. The file format changed.")
    names = lines[hdr_i].lstrip("*").split()
    n_num = len(names) - 2                     # every column after Dot# and Name is a single token
    rows = []
    for l in lines[hdr_i + 1:]:
        t = l.split()
        if len(t) >= len(names):               # station names can contain spaces ("Dover, Delaware")
            rows.append([t[0], " ".join(t[1:len(t) - n_num])] + t[-n_num:])
    df = pd.DataFrame(rows, columns=names)
    need = ["Dot#", "Ref_Nlat", "Ref_Elong", "dN/dt", "dE/dt", "SNd", "SEd", "first_epoch", "last_epoch"]
    missing = [c for c in need if c not in df]
    if missing:
        raise RuntimeError(f"Velocity file lacks columns {missing}. Header: {names}")
    for c in need[1:7]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["t0"] = pd.to_datetime(df["first_epoch"].str[:14], format="%Y%m%d%H%M%S", errors="coerce")
    df["t1"] = pd.to_datetime(df["last_epoch"].str[:14], format="%Y%m%d%H%M%S", errors="coerce")
    # A station has one line per position segment (split at antenna changes or earthquakes). The segments
    # share one velocity, so keep one line per station and measure the record over all its segments.
    g = df.sort_values("t1").groupby("Dot#")
    st = g.last()
    st["dur"] = (g["t1"].max() - g["t0"].min()).dt.days / 365.25
    out = pd.DataFrame({"sta": st.index, "lat": st["Ref_Nlat"], "lon": st["Ref_Elong"],
                        "ve": st["dE/dt"], "vn": st["dN/dt"], "se": st["SEd"], "sn": st["SNd"],
                        "dur": st["dur"]}).reset_index(drop=True).dropna()
    if out.empty:
        raise RuntimeError(f"No usable rows in {url}. Header: {names}")
    if out["vn"].abs().median() > 0.5:
        raise RuntimeError("Velocities look like mm/yr, but m/yr is expected. Check the file's units.")
    return out

MIDAS_COLS = ["sta", "version", "t0", "t1", "dur", "n_epochs", "n_good", "n_pairs",
              "ve", "vn", "vu", "se", "sn", "su", "oe", "on", "ou", "fe", "fn", "fu",
              "sde", "sdn", "sdu", "n_steps", "lat", "lon", "h"]

def load_midas_velocities(url):
    """NGL MIDAS velocities. Column layout is documented in midas.readme.txt at NGL. Velocities in m/yr."""
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    mid = pd.read_csv(io.StringIO(r.text), sep=r"\s+", header=None, comment="#", engine="python")
    if mid.shape[1] < 27:
        raise RuntimeError(f"MIDAS file has {mid.shape[1]} columns, expected 27. Check MIDAS_URL.")
    mid = mid.iloc[:, :27]
    mid.columns = MIDAS_COLS
    return mid[["sta", "lat", "lon", "ve", "vn", "se", "sn", "dur"]]

if GNSS_SOURCE == "earthscope":
    vel = load_earthscope_velocities(EARTHSCOPE_VEL_URL)
elif GNSS_SOURCE == "midas":
    vel = load_midas_velocities(MIDAS_URL)
else:
    raise ValueError(f"Unknown GNSS_SOURCE {GNSS_SOURCE!r}. Use 'earthscope' or 'midas'.")
vel["lon"] = ((vel["lon"] + 180.0) % 360.0) - 180.0   # NOTE: some files list longitudes outside -180..180
if not vel["lat"].between(-90, 90).all():
    raise RuntimeError("The latitude column holds values outside -90..90. The file format changed.")
print(f"{len(vel):,} stations in the {GNSS_SOURCE} velocity file")

lon0, lat0, lon1, lat1 = AOI_BUF_LONLAT
gn = vel[vel.lon.between(lon0, lon1) & vel.lat.between(lat0, lat1)
         & (vel.dur >= GNSS_MIN_YEARS)
         & (vel.se * 1e3 <= GNSS_MAX_SIGMA_MM) & (vel.sn * 1e3 <= GNSS_MAX_SIGMA_MM)].copy()
gn["x"], gn["y"] = to_utm.transform(gn.lon.values, gn.lat.values)
print(f"{len(gn)} GNSS stations after filtering")
if len(gn) < 10:
    raise RuntimeError("Fewer than 10 GNSS stations. Check the AOI, the velocity file, and the filters.")

#### What the next cell does

It defines `strain_rate_field`, which estimates how fast the ground is being stretched or squeezed, from the station velocities. It only defines the function; the next cell uses it.

1. **Units.** It converts the coordinates to kilometres. A velocity change in mm/yr per km is then 10⁻⁶ per year, or 1000 nanostrain per year.
2. **Smooth velocity surface.** `RBFInterpolator` with a thin-plate spline fits a smooth surface through the station velocities, like bending a thin metal sheet so that it passes close to every point. `smoothing` lets the surface pass near the points rather than exactly through them, which reduces the effect of noisy stations. It does this separately for the east and the north velocity, on the coarse grid `gx`, `gy`.
3. **Gradients.** `np.gradient` gives how each velocity changes with distance, east–west and north–south. From these it computes:
   - `exx`: stretching in the east–west direction.
   - `eyy`: stretching in the north–south direction.
   - `exy`: shear, the part of the motion that distorts a square into a diamond.
4. **Outputs**, in nanostrain per year:
   - **`dilatation`** = `exx + eyy`: whether the area is growing (positive, extension) or shrinking (negative). Extension is typical of the normal faulting in the Basin and Range, and it helps fluids flow along faults.
   - **`max_shear`**: the strength of shearing, as on strike-slip faults such as the Walker Lane.
   - **`second_inv`**: the overall strain rate, whatever its style.

In [ ]:
def strain_rate_field(xs, ys, ve_mm, vn_mm, gx, gy, smoothing):
    """Strain rate (nanostrain/yr) from station velocities (mm/yr).

    Coordinates go to km, so gradients are (mm/yr)/km = 1e-6 /yr = 1000 nanostrain/yr.
    gy runs from north to south (image order). np.gradient uses the real coordinates,
    so the signs are correct.
    """
    pts = np.column_stack([xs, ys]) / 1e3
    XX, YY = np.meshgrid(gx, gy)
    q = np.column_stack([XX.ravel(), YY.ravel()]) / 1e3
    fe = RBFInterpolator(pts, ve_mm, kernel="thin_plate_spline", smoothing=smoothing)(q).reshape(XX.shape)
    fn = RBFInterpolator(pts, vn_mm, kernel="thin_plate_spline", smoothing=smoothing)(q).reshape(XX.shape)
    dve_dy, dve_dx = np.gradient(fe, gy / 1e3, gx / 1e3)
    dvn_dy, dvn_dx = np.gradient(fn, gy / 1e3, gx / 1e3)
    s = 1e3
    exx, eyy, exy = dve_dx * s, dvn_dy * s, 0.5 * (dve_dy + dvn_dx) * s
    return {"dilatation": exx + eyy,
            "max_shear": np.sqrt(((exx - eyy) / 2) ** 2 + exy ** 2),
            "second_inv": np.sqrt(exx ** 2 + eyy ** 2 + 2 * exy ** 2)}

#### What the next cell does

It computes the strain rate layers and puts them onto the 100 m grid.

1. **Coarse grid.** Evaluating the spline at every 100 m cell would take millions of calculations. The cell uses a grid with a 2 km step instead (`GNSS_STRAIN_STEP_M`). The `y` values run from north to south, the same order as the rows of an image.
2. **Strain.** It calls `strain_rate_field` with the station positions and their velocities, converted from m/yr to mm/yr.
3. **Distance to stations.** A KD-tree, a structure for fast nearest-neighbor search, finds the distance from each coarse grid point to its nearest station.
4. **Mask.** Points more than `GNSS_MAX_GAP_M` (50 km) from any station are set to NaN, because no measurement constrains the strain there.
5. **Refine.** It interpolates each coarse layer linearly onto the 100 m template and stores it as `gnss_dilatation`, `gnss_max_shear`, or `gnss_second_inv`.
6. It prints the median value of each layer as a quick check. Across the Great Basin, tens of nanostrain per year is typical.

In [ ]:
step = GNSS_STRAIN_STEP_M
gx = np.arange(xmin, xmax + step, step)
gy = np.arange(ymax, ymin - step, -step)
strain = strain_rate_field(gn.x.values, gn.y.values, gn.ve.values * 1e3, gn.vn.values * 1e3,
                           gx, gy, GNSS_RBF_SMOOTHING)
gap, _ = cKDTree(np.column_stack([gn.x, gn.y])).query(
    np.column_stack([a.ravel() for a in np.meshgrid(gx, gy)]))
gap = gap.reshape(len(gy), len(gx))
for k, v in strain.items():
    coarse = xr.DataArray(np.where(gap <= GNSS_MAX_GAP_M, v, np.nan), coords={"y": gy, "x": gx}, dims=("y", "x"))
    fine = coarse.interp(x=template.x, y=template.y, method="linear").astype("float32")
    layers[f"gnss_{k}"] = template.copy(data=fine.values).rename(f"gnss_{k}")
    print(f"  gnss_{k}: median {np.nanmedian(fine.values):.1f} nanostrain/yr")

## 6b. Magnetotelluric conductivity

Magnetotellurics (MT) measures natural electric and magnetic fields at the surface to image how well the rock conducts electricity at depth. Conductive zones in the middle crust are often interpreted as fluids or partial melt, and are associated with geothermal systems in the Great Basin. The EMC models are 3D grids with 0.1° cells, but their real resolution is set by the MT station spacing (about 70 km for the USArray stations). They are regional context, not a fault map. Notebook 03 therefore uses these layers only to rank candidates.

The layer stores log10 **conductivity** (S/m), so higher values mean more conductive. Resistivity in ohm-m is 10 to the power of minus the layer value.

#### What the next cell does

It adds horizontal slices of a 3D electrical conductivity model as layers. No login is needed.

**`mt_conductivity_slices(model_file, depths_km)`:**
1. **Download.** It downloads the model's NetCDF file from the EMC archive into the cache, unless it's already there. The CONUS-MT-2026 file is about 13 MB.
2. **Check.** It opens the file and checks that it holds `log_10sigma` (log10 conductivity) on `depth`, `latitude`, and `longitude` dimensions. Some catalog models store plain conductivity instead, and the cell stops for those.
3. **Arrange.** It converts the longitudes into the range −180 to 180, and sorts the rows from north to south, the order that rioxarray expects.
4. **Slice.** For each depth in `MT_DEPTHS_KM`, it takes the model layer nearest that depth. It warns if the nearest layer is more than 25% away. The model's layers aren't evenly spaced; for example, 10 km becomes 9.42 km in CONUS-MT-2026.
5. **Resample.** It labels the slice as longitude/latitude (EPSG:4326) and resamples it onto the 100 m template with bilinear interpolation. The model cells are about 10 km across, so interpolating gives a smooth surface rather than 10 km blocks. This doesn't add detail: the model's real resolution is tens of kilometres.

**The loop** stores each slice as `mt_log10sigma_<depth>km`. It prints the depth actually used, the range of values (also given as resistivity in ohm-m), and the share of the grid covered.

The layer isn't a fault indicator. Notebook 03 leaves it out of the fault model and uses it only in the candidate ranking, where conductors (high values) count as more favorable.

In [ ]:
def mt_conductivity_slices(model_file, depths_km):
    """Depth slices of an EMC 3D conductivity model as log10 conductivity (S/m), resampled onto the grid.

    Returns {layer_name: (DataArray on the template grid, model depth in km actually used)}.
    """
    local = download(f"{EMC_NETCDF_URL}/{model_file}", CACHE_DIR / "emc" / model_file)
    model = xr.open_dataset(local)
    if "log_10sigma" not in model:
        raise RuntimeError(f"{model_file} has no log_10sigma variable (it has {list(model.data_vars)}). "
                           "Choose a model that stores log10 conductivity.")
    v = model["log_10sigma"]
    for dim in ("depth", "latitude", "longitude"):
        if dim not in v.dims:
            raise RuntimeError(f"{model_file}: expected dimension {dim!r}, found {v.dims}.")
    # Longitudes to -180..180, and rows from north to south (image order) for rioxarray.
    v = v.assign_coords(longitude=((v["longitude"] + 180.0) % 360.0) - 180.0)
    v = v.sortby("longitude").sortby("latitude", ascending=False)
    out = {}
    for d in depths_km:
        s = v.sel(depth=d, method="nearest")
        used = float(s["depth"])
        if abs(used - d) > 0.25 * d:
            warnings.warn(f"{model_file}: nearest model depth to {d} km is {used:.2f} km.")
        s = (s.drop_vars("depth").rename({"longitude": "x", "latitude": "y"}).astype("float32")
               .rio.write_crs("EPSG:4326").rio.write_nodata(np.nan))
        # NOTE: bilinear, because the model cells (0.1 degrees, about 10 km) are far coarser than the grid.
        g = s.rio.reproject_match(template, resampling=Resampling.bilinear)
        g = g.assign_coords(x=template.x, y=template.y)
        out[f"mt_log10sigma_{d:g}km"] = (g, used)
    return out

if MT_MODEL_FILE:
    for name, (da, used) in mt_conductivity_slices(MT_MODEL_FILE, MT_DEPTHS_KM).items():
        layers[name] = template.copy(data=da.values).rename(name)
        v = da.values
        print(f"  {name}: model depth {used:.2f} km, log10 conductivity {np.nanmin(v):.2f} to {np.nanmax(v):.2f} "
              f"(resistivity {10 ** -np.nanmax(v):.2g} to {10 ** -np.nanmin(v):.2g} ohm-m), "
              f"{np.isfinite(v).mean():.1%} of the grid covered")
else:
    print("MT_MODEL_FILE is None: no magnetotelluric layers.")

## 7. Write the feature stack and the labels

#### What the next cell does

It writes the two main outputs.

1. **Feature stack.** It combines all layers into one xarray `Dataset` and stores them as 32-bit floats.
2. **Valid mask.** A cell is **valid** when at least one GeoDAWN layer has data there, so the mask outlines the survey footprint. Notebook 03 trains and predicts only inside it.
3. **Metadata.** It records the coordinate system, and stores the grid definition and the list of GeoDAWN layers as attributes, so that later notebooks can read them.
4. **Write `features.zarr`.** Zarr is a storage format for large arrays. It saves them in pieces, called chunks (here 1024 × 1024 cells), inside a folder, so later notebooks can read part of an array without loading all of it. `mode="w"` replaces any earlier version.
5. **Write `labels.zarr`**, with three layers:
   - `fault`: all fault sources.
   - `fault_qfaults`: Quaternary faults only.
   - `fault_dist_m`: the distance to the nearest fault.
6. It prints the number of layers, the valid area, and the number of fault cells inside it.

In [ ]:
ds = xr.Dataset({k: v.astype("float32") for k, v in layers.items()})
valid = np.zeros((H, W), bool)
for n in geodawn_names:
    valid |= np.isfinite(ds[n].values)
ds["valid"] = (("y", "x"), valid.astype("uint8"))
ds = ds.rio.write_crs(CRS)
ds.attrs.update({"grid": json.dumps(GRID), "geodawn_layers": json.dumps(geodawn_names)})
ds.chunk({"y": 1024, "x": 1024}).to_zarr(store_path("features.zarr"), mode="w", consolidated=True)

lab = xr.Dataset({"fault": (("y", "x"), fault_all), "fault_qfaults": (("y", "x"), fault_q),
                  "fault_dist_m": (("y", "x"), fault_dist_m)},
                 coords={"y": template.y, "x": template.x}).rio.write_crs(CRS)
lab.attrs["grid"] = json.dumps(GRID)
lab.chunk({"y": 1024, "x": 1024}).to_zarr(store_path("labels.zarr"), mode="w", consolidated=True)

print(f"features.zarr: {len(layers)} layers, valid area {valid.mean():.1%} of the grid")
print(f"labels.zarr:   {int((fault_all & valid).sum()):,} fault cells inside the valid mask")

#### What the next cell does

It draws a quick visual check. The figure shows four layers, with the mapped faults drawn as thin white lines:

- the first two GeoDAWN layers,
- the shallow earthquake density,
- the GNSS dilatation.

The colors are stretched from the 2nd to the 98th percentile of each layer, so that a few extreme values don't wash out the rest of the map.

What to look for:
- **Placement.** The layers should cover the survey area and not be blank.
- **Alignment.** The fault lines should fall in sensible places relative to the layers. A shift between the layers and the faults would suggest a coordinate problem.

In [ ]:
show = geodawn_names[:2] + ["eq_logdens_shallow", "gnss_dilatation"]
fig, axes = plt.subplots(1, len(show), figsize=(5 * len(show), 5), constrained_layout=True)
for ax, n in zip(np.atleast_1d(axes), show):
    v = ds[n].values
    lo, hi = np.nanpercentile(v, [2, 98])
    ax.imshow(v, vmin=lo, vmax=hi, cmap="viridis")
    ax.contour(fault_all, levels=[0.5], colors="w", linewidths=0.3)
    ax.set_title(n)
    ax.set_axis_off()